# Deliverable 1 – Data Preprocessing & UNet Baseline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lukas-sek/Pose_Estimation/blob/main/depth_estimation_d1.ipynb)

## Requirements (from spec)

### Preprocessing
- Images in the zip are already **cropped** (square, subject-centred, 10 px margin, bad frames filtered) but **not yet resized**.
- Resize to **256×256** or **384×384** before feeding the network.
- Compare **nearest-neighbour vs bilinear** interpolation for depth map resizing.

### Baseline model
- Lightweight **UNet** (pure PyTorch) trained with **MSE loss**.

### Explorations
- Resolution: **256 vs 384**
- Data augmentation: **horizontal flip, shift, scale/crop, rotation, shearing**
- **Learning rate vs batch size** trade-off

### Metrics
- **RMSE** (foreground pixels)
- **Mean angular error** (normals from depth gradients)
- FPS, parameter count, GPU memory

### Discussion questions
- What makes monocular depth estimation ill-posed?
- Compare the results across all explorations.

## 1 – Setup

In [ ]:
import os
if not os.path.exists('/content/Pose_Estimation'):
    !git clone https://github.com/lukas-sek/Pose_Estimation.git /content/Pose_Estimation
%cd /content/Pose_Estimation

In [ ]:
!pip install -q opencv-python-headless

In [ ]:
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

In [ ]:
import zipfile, os

ZIP_PATH   = '/gdrive/MyDrive/OR/Deliverable 3/preprocessed_data.zip'
EXTRACT_TO = '/content/Pose_Estimation/cloth3d/data/preprocessed_data'

if os.path.isdir(EXTRACT_TO) and os.listdir(EXTRACT_TO):
    print(f'Already extracted at {EXTRACT_TO} – skipping.')
else:
    print(f'Extracting {ZIP_PATH} → {EXTRACT_TO} …')
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        members = zf.namelist()
        prefix = ''
        if members and all(m.startswith(members[0].split('/')[0] + '/') for m in members if '/' in m):
            prefix = members[0].split('/')[0] + '/'
        for member in members:
            stripped = member[len(prefix):]
            if not stripped:
                continue
            dest = os.path.join(EXTRACT_TO, stripped)
            if member.endswith('/'):
                os.makedirs(dest, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                with zf.open(member) as src, open(dest, 'wb') as dst:
                    dst.write(src.read())
    print('Done.')

for entry in sorted(os.listdir(EXTRACT_TO)):
    p = f'{EXTRACT_TO}/{entry}'
    n = len(os.listdir(p)) if os.path.isdir(p) else ''
    print(f'  {entry}/  {n}')

In [ ]:
import os, gc, pickle, time, random
import numpy as np
import cv2
from matplotlib import pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ROOT   = '/content/Pose_Estimation/cloth3d/data/preprocessed_data'
print(f'Device : {DEVICE}')

## 2 – Exploratory Data Analysis

Inspect the raw (pre-resized) data in the zip before touching it.

In [ ]:
def read_list(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

train_list = read_list(f'{ROOT}/train.txt')
val_list   = read_list(f'{ROOT}/validation.txt')
test_list  = read_list(f'{ROOT}/test.txt')
all_list   = train_list + val_list + test_list

print(f'Train : {len(train_list)}')
print(f'Val   : {len(val_list)}')
print(f'Test  : {len(test_list)}')
print(f'Total : {len(all_list)}')
print(f'\nFirst 5 names: {all_list[:5]}')

In [ ]:
# Check raw image & depth shapes and depth value ranges across a sample
sample_names = random.sample(all_list, min(20, len(all_list)))

img_shapes, dpt_shapes, dpt_mins, dpt_maxs, fg_ratios = [], [], [], [], []

for name in sample_names:
    img = cv2.imread(f'{ROOT}/image/{name}.jpg')
    dpt = np.load(f'{ROOT}/depth/{name}.npy')
    mask = dpt > 0
    img_shapes.append(img.shape)
    dpt_shapes.append(dpt.shape)
    if mask.any():
        dpt_mins.append(dpt[mask].min())
        dpt_maxs.append(dpt[mask].max())
    fg_ratios.append(mask.mean())

unique_img = set(img_shapes)
unique_dpt = set(dpt_shapes)
print(f'Unique image shapes : {unique_img}')
print(f'Unique depth shapes : {unique_dpt}')
print(f'Depth range (fg)    : [{np.min(dpt_mins):.3f}, {np.max(dpt_maxs):.3f}] m')
print(f'Foreground ratio    : {np.mean(fg_ratios)*100:.1f}% ± {np.std(fg_ratios)*100:.1f}%')

In [ ]:
# Visualise 6 random samples: RGB | raw depth | depth histogram
viz_names = random.sample(all_list, 6)

fig, axes = plt.subplots(6, 3, figsize=(13, 18))
fig.suptitle('EDA – Raw samples (pre-resize)', fontweight='bold', fontsize=13)

for row, name in enumerate(viz_names):
    img = cv2.cvtColor(cv2.imread(f'{ROOT}/image/{name}.jpg'), cv2.COLOR_BGR2RGB)
    dpt = np.load(f'{ROOT}/depth/{name}.npy')
    mask = dpt > 0

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f'{name}  {img.shape[:2]}', fontsize=8)
    axes[row, 0].axis('off')

    im = axes[row, 1].imshow(dpt, cmap='plasma')
    axes[row, 1].set_title(f'depth  fg={mask.mean()*100:.0f}%', fontsize=8)
    axes[row, 1].axis('off')
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046)

    if mask.any():
        axes[row, 2].hist(dpt[mask].ravel(), bins=40, color='steelblue', edgecolor='none')
    axes[row, 2].set_title('depth histogram (fg)', fontsize=8)
    axes[row, 2].set_xlabel('depth (m)')

plt.tight_layout()
plt.savefig('/content/eda_samples.png', dpi=120)
plt.show()

In [ ]:
# Compare nearest-neighbour vs bilinear resizing for a depth map
# Bilinear can bleed zero (background) values into foreground edges
name = train_list[0]
dpt  = np.load(f'{ROOT}/depth/{name}.npy')

dpt_nn  = cv2.resize(dpt, (256, 256), interpolation=cv2.INTER_NEAREST)
dpt_bil = cv2.resize(dpt, (256, 256), interpolation=cv2.INTER_LINEAR)

diff = np.abs(dpt_nn.astype(np.float32) - dpt_bil.astype(np.float32))

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Depth resize: nearest-neighbour vs bilinear', fontweight='bold')
axes[0].imshow(dpt,     cmap='plasma'); axes[0].set_title(f'Original {dpt.shape}');  axes[0].axis('off')
axes[1].imshow(dpt_nn,  cmap='plasma'); axes[1].set_title('Nearest 256');            axes[1].axis('off')
axes[2].imshow(dpt_bil, cmap='plasma'); axes[2].set_title('Bilinear 256');           axes[2].axis('off')
im = axes[3].imshow(diff, cmap='hot');  axes[3].set_title('|NN − Bilinear|');        axes[3].axis('off')
plt.colorbar(im, ax=axes[3], fraction=0.046)
plt.tight_layout()
plt.savefig('/content/eda_interp.png', dpi=120)
plt.show()

# Boundary artefact: how many zero pixels appear on edges after bilinear?
fg_orig = (dpt > 0).mean()
fg_nn   = (dpt_nn > 0).mean()
fg_bil  = (dpt_bil > 0).mean()
print(f'Foreground ratio  original: {fg_orig*100:.1f}%')
print(f'Foreground ratio  NN      : {fg_nn*100:.1f}%')
print(f'Foreground ratio  Bilinear: {fg_bil*100:.1f}%  (bleeding if lower than NN)')

## 3 – Configuration

In [ ]:
CONFIG = {
    'root'         : ROOT,
    # Resize resolution – explore 256 vs 384
    'img_size'     : 256,
    # Depth interpolation for resize: 'nearest' | 'bilinear'
    'depth_interp' : 'nearest',

    # Augmentation toggles
    'aug_hflip'    : True,
    'aug_shift'    : True,
    'aug_scale'    : True,
    'aug_rotate'   : True,
    'aug_shear'    : True,

    # Training
    'batch_size'   : 8,
    'lr'           : 1e-3,
    'weight_decay' : 1e-4,
    'epochs'       : 30,
    'patience'     : 5,

    'checkpoint'   : '/content/best_unet_d1.pth',
    'history_path' : '/content/history_d1.pkl',
}
print(CONFIG)

## 3b – Pre-resize to Disk (run once)

Resizes every image and depth map to 256×256 and saves them to a fast local cache.  
This runs once (~2–5 min) and makes every subsequent epoch ~50× faster.

In [ ]:
import shutil

TARGET_SIZE = 256   # change to 384 for that exploration

def resize_inplace(root, size):
    """
    Resize every image + depth map to (size x size) IN PLACE.
    Depth saved as float16 (~8x smaller than original float32 at full res).
    Originals are overwritten, not duplicated — no extra disk space needed.
    """
    img_dir = f'{root}/image'
    dpt_dir = f'{root}/depth'

    img_files = sorted(f for f in os.listdir(img_dir) if f.endswith('.jpg'))

    # Check if already done (first file already at target size)
    probe = cv2.imread(f'{img_dir}/{img_files[0]}')
    if probe.shape[0] == size and probe.shape[1] == size:
        print(f'Already at {size}×{size} – skipping resize.')
        return

    print(f'Resizing {len(img_files)} samples to {size}×{size} in place …')
    print(f'(depth saved as float16 to save ~8x disk space)')
    for i, fname in enumerate(img_files):
        stem = fname.rsplit('.', 1)[0]

        # RGB
        img = cv2.imread(f'{img_dir}/{fname}')
        img = cv2.resize(img, (size, size), interpolation=cv2.INTER_LINEAR)
        cv2.imwrite(f'{img_dir}/{fname}', img, [cv2.IMWRITE_JPEG_QUALITY, 95])

        # Depth: nearest-neighbour + save as float16
        dpt = np.load(f'{dpt_dir}/{stem}.npy').astype(np.float32)
        dpt = cv2.resize(dpt, (size, size), interpolation=cv2.INTER_NEAREST)
        np.save(f'{dpt_dir}/{stem}.npy', dpt.astype(np.float16))

        if (i + 1) % 1000 == 0:
            print(f'  {i+1}/{len(img_files)}')

    print(f'Done. All files resized to {size}×{size}.')


resize_inplace(ROOT, TARGET_SIZE)

# Verify disk usage after
import shutil
total, used, free = shutil.disk_usage('/')
print(f'\nDisk: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total  ({free/1e9:.1f} GB free)')

## 4 – Dataset & Augmentation

The zip images are already cropped (square, centred, 10 px margin). The only remaining preprocessing here is **resizing** to the target resolution.

Augmentations applied to training data only:
- Horizontal flip
- Random shift (±10% of image size)
- Random scale/crop (zoom in 80–100% then pad back)
- Random rotation (±15°)
- Random shearing (±10°)

In [ ]:
class Cloth3DDataset(Dataset):
    def __init__(self, data_list, root, img_size=256, augment=False, cfg=None):
        self.data_list = data_list
        self.root      = root
        self.img_size  = img_size
        self.augment   = augment
        self.cfg       = cfg or {}

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        name = self.data_list[idx]
        s    = self.img_size

        # float16 on disk → float32 in memory
        dpt  = np.load(f'{self.root}/depth/{name}.npy').astype(np.float32)
        img  = cv2.cvtColor(cv2.imread(f'{self.root}/image/{name}.jpg'),
                            cv2.COLOR_BGR2RGB).astype(np.float32)
        mask = dpt > 0

        if self.augment:
            img, dpt = self._augment(img, dpt, s)
            mask = dpt > 0

        if mask.any():
            dpt[mask] = (dpt[mask] - dpt[mask].min() + 0.001) / 2.0

        if mask.any():
            for c in range(3):
                ch = img[:, :, c]
                mu, sigma = ch[mask].mean(), ch[mask].std()
                img[:, :, c] = (ch - mu) / (sigma + 1e-5)

        img = torch.from_numpy(img).permute(2, 0, 1)
        dpt = torch.from_numpy(dpt).unsqueeze(0)
        return img, dpt

    def _augment(self, img, dpt, s):
        cfg = self.cfg

        if cfg.get('aug_hflip', True) and random.random() > 0.5:
            img = img[:, ::-1, :].copy()
            dpt = dpt[:, ::-1].copy()

        M = np.eye(3, dtype=np.float32)
        cx, cy = s / 2, s / 2

        if cfg.get('aug_rotate', True):
            rad = np.deg2rad(random.uniform(-15, 15))
            R   = np.array([[np.cos(rad), -np.sin(rad), 0],
                            [np.sin(rad),  np.cos(rad), 0],
                            [0, 0, 1]], dtype=np.float32)
            T1  = np.array([[1,0,-cx],[0,1,-cy],[0,0,1]], dtype=np.float32)
            T2  = np.array([[1,0, cx],[0,1, cy],[0,0,1]], dtype=np.float32)
            M   = T2 @ R @ T1 @ M

        if cfg.get('aug_shear', True):
            shx = np.deg2rad(random.uniform(-10, 10))
            shy = np.deg2rad(random.uniform(-10, 10))
            Sh  = np.array([[1, np.tan(shx), 0],
                            [np.tan(shy), 1, 0],
                            [0, 0, 1]], dtype=np.float32)
            M   = Sh @ M

        if cfg.get('aug_shift', True):
            M[0, 2] += random.uniform(-0.1, 0.1) * s
            M[1, 2] += random.uniform(-0.1, 0.1) * s

        aff = M[:2]
        img = cv2.warpAffine(img, aff, (s, s), flags=cv2.INTER_LINEAR,
                             borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        dpt = cv2.warpAffine(dpt, aff, (s, s), flags=cv2.INTER_NEAREST,
                             borderMode=cv2.BORDER_CONSTANT, borderValue=0)

        if cfg.get('aug_scale', True):
            scale = random.uniform(0.8, 1.0)
            crop  = int(s * scale)
            x0    = random.randint(0, s - crop)
            y0    = random.randint(0, s - crop)
            img   = cv2.resize(img[y0:y0+crop, x0:x0+crop], (s, s), interpolation=cv2.INTER_LINEAR)
            dpt   = cv2.resize(dpt[y0:y0+crop, x0:x0+crop], (s, s), interpolation=cv2.INTER_NEAREST)

        return img, dpt

## 5 – UNet Architecture

A lightweight pure-PyTorch UNet with skip connections. Five encoder levels + bottleneck + symmetric decoder.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.GELU(),
        )
    def forward(self, x): return self.block(x)


class UNet(nn.Module):
    """
    Lightweight UNet matching the baseline spec:
    channels [64, 128, 256, 512, 1024] with max-pool down and bilinear up.
    """
    def __init__(self, in_ch=3, out_ch=1, features=(64, 128, 256, 512, 1024)):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools    = nn.ModuleList()
        ch = in_ch
        for f in features[:-1]:
            self.encoders.append(ConvBlock(ch, f))
            self.pools.append(nn.MaxPool2d(2))
            ch = f

        self.bottleneck = ConvBlock(ch, features[-1])

        self.upconvs  = nn.ModuleList()
        self.decoders = nn.ModuleList()
        rev = list(reversed(features[:-1]))
        ch  = features[-1]
        for f in rev:
            self.upconvs.append(nn.ConvTranspose2d(ch, f, 2, stride=2))
            self.decoders.append(ConvBlock(f * 2, f))
            ch = f

        self.head = nn.Sequential(nn.Conv2d(ch, out_ch, 1), nn.Sigmoid())

    def forward(self, x):
        skips = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x)
            skips.append(x)
            x = pool(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.upconvs, self.decoders, reversed(skips)):
            x = up(x)
            if x.shape != skip.shape:
                x = F.interpolate(x, size=skip.shape[-2:], mode='bilinear', align_corners=False)
            x = dec(torch.cat([skip, x], dim=1))
        return self.head(x)

## 6 – Metrics

In [ ]:
def rmse(pred, gt):
    mask = gt > 0
    if not mask.any():
        return torch.tensor(0.0, device=pred.device)
    return torch.sqrt(F.mse_loss(pred[mask], gt[mask]))


def depth_to_normals(depth, eps=1e-6):
    dz_dx = depth[:, :, :, 2:] - depth[:, :, :, :-2]
    dz_dy = depth[:, :, 2:, :] - depth[:, :, :-2, :]
    dz_dx = F.pad(dz_dx, [1, 1, 0, 0])
    dz_dy = F.pad(dz_dy, [0, 0, 1, 1])
    n = torch.cat([-dz_dx, -dz_dy, torch.ones_like(dz_dx)], dim=1)
    return n / (torch.norm(n, dim=1, keepdim=True) + eps)


def mean_angular_error(pred_depth, gt_depth):
    mask    = (gt_depth > 0).squeeze(1)
    n_pred  = depth_to_normals(pred_depth)
    n_gt    = depth_to_normals(gt_depth)
    cos_sim = (n_pred * n_gt).sum(dim=1).clamp(-1.0, 1.0)
    angle   = torch.acos(cos_sim) * (180.0 / torch.pi)
    if not mask.any():
        return torch.tensor(0.0, device=pred_depth.device)
    return angle[mask].mean()

## 7 – Training

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for X, Y in loader:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        loss = F.mse_loss(model(X), Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def val_epoch(model, loader):
    model.eval()
    tot_rmse, tot_mae, n = 0.0, 0.0, 0
    for X, Y in loader:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        pred = model(X)
        tot_rmse += rmse(pred, Y).item()
        tot_mae  += mean_angular_error(pred, Y).item()
        n += 1
    return tot_rmse / n, tot_mae / n


def train(cfg, train_list, val_list, model=None):
    s = cfg['img_size']
    interp = cfg.get('depth_interp', 'nearest')

    tr_ds = Cloth3DDataset(train_list, cfg['root'], s, interp, augment=True,  cfg=cfg)
    vl_ds = Cloth3DDataset(val_list,   cfg['root'], s, interp, augment=False, cfg=cfg)
    tr_dl = DataLoader(tr_ds, batch_size=cfg['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    vl_dl = DataLoader(vl_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    if model is None:
        model = UNet().to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'])

    best_rmse, patience_ctr = float('inf'), 0
    history = {'train_loss': [], 'val_rmse': [], 'val_mae_n': []}

    for epoch in range(cfg['epochs']):
        t0 = time.time()
        tr_loss        = train_epoch(model, tr_dl, optimizer)
        v_rmse, v_mae  = val_epoch(model, vl_dl)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_rmse'  ].append(v_rmse)
        history['val_mae_n' ].append(v_mae)

        print(f'Epoch {epoch+1:3d}/{cfg["epochs"]} | '
              f'loss={tr_loss:.4f} | val_RMSE={v_rmse:.4f} | '
              f'val_NormalMAE={v_mae:.2f}° | {time.time()-t0:.1f}s')

        if v_rmse < best_rmse:
            best_rmse = v_rmse
            torch.save(model.state_dict(), cfg['checkpoint'])
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= cfg['patience']:
                print(f'Early stopping at epoch {epoch+1}')
                break

    with open(cfg['history_path'], 'wb') as fh:
        pickle.dump(history, fh)

    print(f'Best val RMSE: {best_rmse:.4f}')
    return model, history

## 8 – Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, test_list, cfg, label=''):
    model.eval()
    model.load_state_dict(torch.load(cfg['checkpoint'], map_location=DEVICE))

    s      = cfg['img_size']
    interp = cfg.get('depth_interp', 'nearest')
    ds     = Cloth3DDataset(test_list, cfg['root'], s, interp, augment=False)
    dl     = DataLoader(ds, batch_size=1, shuffle=False)

    tot_rmse, tot_mae = 0.0, 0.0
    fps_times = []
    if DEVICE == 'cuda':
        torch.cuda.reset_peak_memory_stats()

    for X, Y in dl:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        t0 = time.time()
        pred = model(X)
        if DEVICE == 'cuda': torch.cuda.synchronize()
        fps_times.append(time.time() - t0)
        tot_rmse += rmse(pred, Y).item()
        tot_mae  += mean_angular_error(pred, Y).item()

    n       = len(dl)
    params  = sum(p.numel() for p in model.parameters())
    gpu_mb  = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE == 'cuda' else 0
    fps     = 1.0 / (sum(fps_times) / len(fps_times))

    print(f'{'='*52}')
    if label: print(f'  {label}')
    print(f'  Test RMSE         : {tot_rmse/n:.4f}')
    print(f'  Normal MAE        : {tot_mae/n:.2f}°')
    print(f'  FPS               : {fps:.1f}')
    print(f'  Parameters        : {params:,}')
    print(f'  Peak GPU mem (MB) : {gpu_mb:.1f}')
    print(f'{'='*52}')

    return dict(label=label, rmse=tot_rmse/n, normal_mae=tot_mae/n,
                fps=fps, params=params, gpu_mb=gpu_mb)

## 9 – Baseline Run (256 px, nearest, all augmentations, lr=1e-3, bs=8)

In [ ]:
model_baseline, hist_baseline = train(CONFIG, train_list, val_list)
res_baseline = evaluate(model_baseline, test_list, CONFIG, label='Baseline 256 NN full-aug')

## 10 – Explorations

Each cell runs one controlled experiment. Uncomment and run one at a time.

### 10a – Resolution: 256 vs 384

In [ ]:
def run_exp(overrides, label):
    tag = label.lower().replace(' ', '_').replace('/', '_')
    cfg = {
        **CONFIG,
        **overrides,
        'checkpoint'  : f'/content/unet_{tag}.pth',
        'history_path': f'/content/hist_{tag}.pkl',
    }
    m, _ = train(cfg, train_list, val_list)
    return evaluate(m, test_list, cfg, label=label)


# Resolution comparison
# r_256 = run_exp({'img_size': 256}, '256 px')
# r_384 = run_exp({'img_size': 384, 'batch_size': 4}, '384 px')

### 10b – Augmentation ablation

In [ ]:
# No augmentation
# r_noaug = run_exp(
#     {'aug_hflip': False, 'aug_shift': False, 'aug_scale': False,
#      'aug_rotate': False, 'aug_shear': False},
#     'No augmentation'
# )

# Flip only
# r_flip = run_exp(
#     {'aug_hflip': True, 'aug_shift': False, 'aug_scale': False,
#      'aug_rotate': False, 'aug_shear': False},
#     'Flip only'
# )

# Full augmentation (same as baseline)
# r_fullaug = res_baseline

### 10c – Learning rate vs batch size

In [ ]:
# lr=1e-3 bs=8  (baseline)
# lr=1e-4 bs=8
# r_lr4 = run_exp({'lr': 1e-4}, 'lr=1e-4 bs=8')

# lr=1e-3 bs=16
# r_bs16 = run_exp({'batch_size': 16}, 'lr=1e-3 bs=16')

# lr=1e-3 bs=4
# r_bs4 = run_exp({'batch_size': 4}, 'lr=1e-3 bs=4')

### 10d – Depth interpolation: nearest vs bilinear

In [ ]:
# r_nn  = res_baseline   # nearest (already done)
# r_bil = run_exp({'depth_interp': 'bilinear'}, '256 bilinear depth')

## 11 – Results Comparison Table

In [ ]:
import pandas as pd

# Fill in after running experiments
# (label, RMSE, NormalMAE°, FPS, Params_M, GPU_MB)
results_table = [
    ('Baseline 256 NN full-aug',   None, None, None, None, None),
    ('384 px',                     None, None, None, None, None),
    ('No augmentation',            None, None, None, None, None),
    ('Flip only',                  None, None, None, None, None),
    ('lr=1e-4 bs=8',               None, None, None, None, None),
    ('lr=1e-3 bs=16',              None, None, None, None, None),
    ('lr=1e-3 bs=4',               None, None, None, None, None),
    ('256 bilinear depth',         None, None, None, None, None),
]

cols = ['Config', 'RMSE ↓', 'NormalMAE° ↓', 'FPS ↑', 'Params (M)', 'GPU MB']
df   = pd.DataFrame(results_table, columns=cols)
print(df.to_string(index=False))

filled = [(r[0], r[1], r[2]) for r in results_table if r[1] is not None]
if filled:
    labels, rmses, maes = zip(*filled)
    x = range(len(labels))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.bar(x, rmses);  ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=35, ha='right'); ax1.set_ylabel('RMSE ↓')
    ax2.bar(x, maes);   ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=35, ha='right'); ax2.set_ylabel('Normal MAE (°) ↓')
    plt.tight_layout(); plt.savefig('/content/d1_results.png', dpi=150); plt.show()
else:
    print('Fill in results_table after running experiments.')

## 12 – Visualisation

In [ ]:
hist = pickle.load(open(CONFIG['history_path'], 'rb'))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('UNet Baseline – Training History', fontweight='bold')
axes[0].plot(hist['train_loss']); axes[0].set(title='Train Loss (MSE)',   xlabel='epoch')
axes[1].plot(hist['val_rmse'  ]); axes[1].set(title='Val RMSE',           xlabel='epoch')
axes[2].plot(hist['val_mae_n' ]); axes[2].set(title='Val Normal MAE (°)', xlabel='epoch')
plt.tight_layout(); plt.savefig('/content/d1_curves.png', dpi=150); plt.show()

In [ ]:
@torch.no_grad()
def visualize_predictions(model, test_list, cfg, n_samples=4):
    model.eval()
    s      = cfg['img_size']
    interp = cfg.get('depth_interp', 'nearest')
    ds     = Cloth3DDataset(test_list, cfg['root'], s, interp, augment=False)

    n_samples = min(n_samples, len(ds))
    if n_samples == 0:
        print('No test samples.'); return

    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle('RGB  |  GT depth  |  Predicted depth  |  Normal error map', fontweight='bold')

    for i in range(n_samples):
        X, Y = ds[i]
        pred = model(X.unsqueeze(0).to(DEVICE)).cpu().squeeze(0)

        rgb     = X.permute(1, 2, 0).numpy()
        rgb     = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-5)
        gt_np   = Y.squeeze().numpy()
        pred_np = pred.squeeze().numpy()

        n_pred  = depth_to_normals(pred.unsqueeze(0))
        n_gt    = depth_to_normals(Y.unsqueeze(0))
        cos_s   = (n_pred * n_gt).sum(dim=1).clamp(-1.0, 1.0)
        err_map = (torch.acos(cos_s) * 180.0 / torch.pi).squeeze().numpy()

        axes[i, 0].imshow(rgb)
        axes[i, 1].imshow(gt_np,   cmap='plasma')
        axes[i, 2].imshow(pred_np, cmap='plasma')
        im = axes[i, 3].imshow(err_map, cmap='hot', vmin=0, vmax=45)
        plt.colorbar(im, ax=axes[i, 3], fraction=0.046)
        for ax in axes[i]: ax.axis('off')

    plt.tight_layout()
    plt.savefig('/content/d1_predictions.png', dpi=150)
    plt.show()


visualize_predictions(model_baseline, test_list, CONFIG)

## 13 – Discussion Questions

### Q1 – What makes monocular depth estimation ill-posed?

*Hints: a single 2D image collapses the 3D world onto a flat sensor — infinite 3D scenes can project to the same image (scale-depth ambiguity). Objects of different sizes at different distances look identical. Without stereo disparity, motion parallax, or structured light the depth signal must be inferred purely from learned cues: perspective foreshortening, texture gradient, defocus blur, shading. The problem is fundamentally under-determined and the solution space is continuous.*

**Your answer:**

> *(write here)*

### Q2 – Compare the results across all explorations

*Structure your answer around the three axes:*

**Resolution (256 vs 384):**  
*Does higher resolution improve RMSE? At what cost in FPS and GPU memory? Is the gain worth the 2.25× more pixels?*

**Augmentation:**  
*Compare no-aug vs flip-only vs full-aug. Which augmentation contributes most? Do geometric transforms (rotation, shear) help or hurt given that the body pose is already varied across sequences?*

**Learning rate vs batch size:**  
*Larger batch → sharper gradients but fewer updates per epoch. Lower LR → more stable but slower. Which combination converges fastest and to the lowest RMSE? Does the linear scaling rule (LR ∝ batch size) hold here?*

**Depth interpolation:**  
*Does bilinear bleeding at boundaries noticeably hurt RMSE or normal MAE compared to nearest-neighbour? Reference the EDA interpolation comparison from Section 2.*

**Your answer:**

> *(write here after filling the results table in Section 11)*